# Epsilon Schedule Estimation

Helper to compute per-phase `epsilon_decay` values that take the agent from `epsilon_start` to `epsilon_min` over a target fraction of each phase's gradient updates.

The agent (`src/core/learning_agent.py::DQNAgent.update`) applies the decay once per gradient update. With `update_frequency=4` this is once every 4 environment steps. The inputs below are therefore per-phase **gradient update counts**, not episode counts.

Formulas (one decay step per gradient update, target reached at `portion * total_steps`):

- Exponential (multiplicative): `epsilon_t = epsilon_start * decay^t`, so
  $$\text{decay} = \left(\frac{\epsilon_{min}}{\epsilon_{start}}\right)^{1/(\text{portion} \cdot \text{total\_steps})}$$
- Linear (additive): `epsilon_t = max(epsilon_min, epsilon_start - decay * t)`, so
  $$\text{decay} = \frac{\epsilon_{start} - \epsilon_{min}}{\text{portion} \cdot \text{total\_steps}}$$

The output array maps one-to-one onto the `training.epsilon_decay` list in `experiment_config.yml`.

The numeric inputs below are **illustrative** — adjust for your own dataset size, code, and `update_frequency`.

In [ ]:
import numpy as np

np.set_printoptions(precision=10, suppress=True)


def exponential_epsilon_decay(epsilon_start, epsilon_min, total_steps, portion):
    """Per-update multiplier that lands at epsilon_min after portion*total_steps updates."""
    return (np.asarray(epsilon_min) / np.asarray(epsilon_start)) ** (
        1.0 / (np.asarray(portion) * np.asarray(total_steps))
    )


def linear_epsilon_decay(epsilon_start, epsilon_min, total_steps, portion):
    """Per-update decrement that lands at epsilon_min after portion*total_steps updates."""
    return (np.asarray(epsilon_start) - np.asarray(epsilon_min)) / (
        np.asarray(portion) * np.asarray(total_steps)
    )

## Example: exponential schedule

Six phases, decay reaches `epsilon_min` at 80% of each phase's gradient updates.

In [ ]:
epsilon_start = np.array([1.0, 0.8, 0.6, 0.4, 0.3, 0.2])
epsilon_min   = np.array([0.3, 0.2, 0.1, 0.05, 0.01, 0.01])
total_steps   = np.array([57_600, 116_000, 152_000, 172_000, 189_000, 194_000])
portion       = 0.8

print(exponential_epsilon_decay(epsilon_start, epsilon_min, total_steps, portion))

## Example: linear schedule

Same six phases. The decay values below correspond to the schedule currently shipped in `config/experiments_config/64x128_Full/experiment_config.yml`.

In [ ]:
epsilon_start = np.array([1.0, 0.8, 0.6, 0.6, 0.6, 0.2])
epsilon_min   = np.array([0.3, 0.2, 0.1, 0.1, 0.1, 0.01])
total_steps   = np.array([800_000, 260_000, 260_000, 405_000, 690_000, 330_000])
portion       = 0.95

print(linear_epsilon_decay(epsilon_start, epsilon_min, total_steps, portion))